# Single-Asset Backtesting Lab

This notebook runs reusable single-asset strategies across the anonymized assets with the same 60/40 train-test split.

In [40]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from backtesting import Backtest, Strategy

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

prices = pd.read_csv('prices.csv', parse_dates=['Date']).set_index('Date')
all_assets = list(prices.columns)

split_idx = int(len(prices) * 0.60)
train_prices = prices.iloc[:split_idx].copy()
test_prices = prices.iloc[split_idx:].copy()

pd.Series({
    'Total rows': len(prices),
    'Training rows': len(train_prices),
    'Test rows': len(test_prices),
    'Train start': train_prices.index.min(),
    'Train end': train_prices.index.max(),
    'Test start': test_prices.index.min(),
    'Test end': test_prices.index.max(),
})


## Helpers

In [ ]:
def make_ohlc(close: pd.Series) -> pd.DataFrame:
    close = close.dropna().astype(float)
    data = pd.DataFrame(index=close.index)
    data['Open'] = close
    data['High'] = close
    data['Low'] = close
    data['Close'] = close
    data['Volume'] = 0.0
    return data


## Strategy Definitions

Add or edit single-asset strategy classes here.

In [ ]:
class TimeSeriesMomentumStrategy(Strategy):
    lookback = 60
    vol_window = 40
    risk_budget = 0.25

    def init(self):
        close = pd.Series(self.data.Close.s, index=self.data.index)
        returns = close.pct_change()
        momentum = np.sign(close.pct_change(self.lookback))
        vol = returns.rolling(self.vol_window, min_periods=max(5, self.vol_window // 2)).std()

        self.signal = self.I(lambda: momentum.fillna(0).to_numpy())
        self.vol = self.I(lambda: vol.to_numpy())

    def next(self):
        signal = self.signal[-1]
        vol = self.vol[-1]

        if not np.isfinite(signal) or signal == 0:
            self.position.close()
            return

        size = 0.0 if not np.isfinite(vol) or vol <= 0 else min(self.risk_budget, 0.01 / vol)
        if size <= 0:
            self.position.close()
            return

        if signal > 0:
            if self.position.is_short:
                self.position.close()
            if not self.position.is_long:
                self.buy(size=size)
        else:
            if self.position.is_long:
                self.position.close()
            if not self.position.is_short:
                self.sell(size=size)


class MeanReversionStrategy(Strategy):
    lookback = 20
    vol_window = 40
    entry_z = 1.5
    exit_z = 0.5
    risk_budget = 0.20

    def init(self):
        close = pd.Series(self.data.Close.s, index=self.data.index)
        returns = close.pct_change()
        mean = close.rolling(self.lookback, min_periods=max(5, self.lookback // 2)).mean()
        std = close.rolling(self.lookback, min_periods=max(5, self.lookback // 2)).std()
        zscore = (close - mean) / std.replace(0, np.nan)
        vol = returns.rolling(self.vol_window, min_periods=max(5, self.vol_window // 2)).std()

        self.zscore = self.I(lambda: zscore.to_numpy())
        self.vol = self.I(lambda: vol.to_numpy())

    def next(self):
        zscore = self.zscore[-1]
        vol = self.vol[-1]

        if not np.isfinite(zscore):
            self.position.close()
            return

        size = 0.0 if not np.isfinite(vol) or vol <= 0 else min(self.risk_budget, 0.01 / vol)
        if size <= 0:
            self.position.close()
            return

        if abs(zscore) <= self.exit_z:
            self.position.close()
            return

        if zscore < -self.entry_z:
            if self.position.is_short:
                self.position.close()
            if not self.position.is_long:
                self.buy(size=size)
        elif zscore > self.entry_z:
            if self.position.is_long:
                self.position.close()
            if not self.position.is_short:
                self.sell(size=size)


class YourStrategy(Strategy):
    lookback = 50

    def init(self):
        close = pd.Series(self.data.Close.s, index=self.data.index)
        sma = close.rolling(self.lookback).mean()
        self.sma = self.I(lambda: sma.to_numpy())

    def next(self):
        if not np.isfinite(self.sma[-1]):
            return

        if self.data.Close[-1] > self.sma[-1] and not self.position.is_long:
            self.position.close()
            self.buy(size=0.5)
        elif self.data.Close[-1] < self.sma[-1] and not self.position.is_short:
            self.position.close()
            self.sell(size=0.5)


In [ ]:
STRATEGY_REGISTRY = {
    'time_series': TimeSeriesMomentumStrategy,
    'mean_reversion': MeanReversionStrategy,
    'your_strategy': YourStrategy,
}

STRATEGY_REGISTRY


## Single-Asset Strategy Backtests

In [ ]:
# strategy_name = 'your_strategy'
strategy_name = 'time_series'
# strategy_name = 'mean_reversion'

selected_assets = all_assets
# selected_assets = [c for c in all_assets if c.startswith('Stock_')]
# selected_assets = ['Stock_01', 'Stock_02', 'Idx_01', 'FX_01']

backtest_split = 'test'
# backtest_split = 'train'
# backtest_split = 'full'

price_source = {
    'train': train_prices,
    'test': test_prices,
    'full': prices,
}[backtest_split]

strategy_cls = STRATEGY_REGISTRY[strategy_name]
pd.Series({'strategy': strategy_name, 'split': backtest_split, 'rows': len(price_source)})

In [ ]:
rows = []
backtests = {}

for asset in selected_assets:
    data = make_ohlc(price_source[asset])
    bt = Backtest(
        data,
        strategy_cls,
        cash=100_000,
        commission=0.0,
        hedging=False,
        exclusive_orders=True,
        trade_on_close=False,
        finalize_trades=True,
    )
    stats = bt.run()
    backtests[asset] = (bt, stats)
    rows.append({
        'Asset': asset,
        'Return [%]': stats['Return [%]'],
        'Buy & Hold Return [%]': stats['Buy & Hold Return [%]'],
        'Sharpe Ratio': stats['Sharpe Ratio'],
        'Max. Drawdown [%]': stats['Max. Drawdown [%]'],
        '# Trades': stats['# Trades'],
        'Exposure Time [%]': stats['Exposure Time [%]'],
        'Equity Final [$]': stats['Equity Final [$]'],
    })

summary = pd.DataFrame(rows).sort_values(['Sharpe Ratio', 'Return [%]'], ascending=[False, False])
summary.to_csv(OUTPUT_DIR / f'backtesting_{strategy_name}_{backtest_split}_summary.csv', index=False)
summary.head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

top_sharpe = summary.head(10).sort_values('Sharpe Ratio')
axes[0].barh(top_sharpe['Asset'], top_sharpe['Sharpe Ratio'], color='tab:blue')
axes[0].set_title(f'Top 10 Assets by Sharpe: {strategy_name} ({backtest_split})')
axes[0].set_xlabel('Sharpe Ratio')

top_return = summary.head(10).sort_values('Return [%]')
axes[1].barh(top_return['Asset'], top_return['Return [%]'], color='tab:green')
axes[1].set_title(f'Top 10 Assets by Return: {strategy_name} ({backtest_split})')
axes[1].set_xlabel('Return [%]')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(summary['Buy & Hold Return [%]'], summary['Return [%]'], alpha=0.8)
ax.axhline(0, color='black', linewidth=1)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Buy & Hold Return [%]')
ax.set_ylabel('Strategy Return [%]')
ax.set_title(f'Strategy vs Buy-and-Hold: {strategy_name} ({backtest_split})')
plt.show()

In [ ]:
best_asset = summary.iloc[0]['Asset']
best_bt, best_stats = backtests[best_asset]
best_asset, best_stats[['Return [%]', 'Sharpe Ratio', 'Max. Drawdown [%]', '# Trades']]

In [ ]:
best_bt.plot(filename=str(OUTPUT_DIR / f'backtesting_{strategy_name}_{backtest_split}_{best_asset}.html'), open_browser=False)
f'Saved interactive plot to {OUTPUT_DIR / f"backtesting_{strategy_name}_{backtest_split}_{best_asset}.html"}'